# ST 554 Project 2
*By: Cass Crews*

In this notebook, we will explore the capabilities of the `PySpark` API. More specifically, we will demonstrate the functionality of the `PySpark.sql` and `PySpark.pandas` modules, two key modules that bridge the gap between Python's functionalities and the big data capabilities of Apache Spark. 

In Part I, we will use a custom class, specifically built for this notebook and stored in the same repository, to conduct basic data validation on the SQL-style dataframes central to `PySpark.sql`. 

In Part II, we will complete the same data manipulation process using the `PySpark.sql` and `PySpark.pandas` modules, highlighting their similarities and differences. 

Before we get started, we need to read in the modules and sub-modules we'll use throughout the notebook. we also need to initialize a Spark session. 

In [2]:
# Importing key modules and sub-modules, and initializing a Spark session
import numpy as np
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from functools import reduce
from pyspark.sql.types import *
import pandas as pd
from pyspark.sql import SparkSession
spark = SparkSession.builder.master('local[*]').appName('my_app').getOrCreate()

## Part I

While `PySpark.sql` is incredibly useful for accessing and manipulating databases, some steps of a standard exploratory data analysis (EDA) can be more tedious than they are when working in R's tidyverse or when utilizing the `pandas` module. Thus, Part I explores a custom class, `SparkDataCheck`, created specifically for this notebook to highlight the abilities to use classes and associated methods to simplify common EDA steps. The source script for `SparkDataCheck` can be found [here](https://github.com/jccrews256/ST-554-Project-2/blob/main/SparkDataCheck.py). 

`SparkDataCheck` is designed to be initialized in three convenient ways, with each resulting in a `.df` attribute that stores a dataset in a `PySpark.sql` dataframe:

- `.__init__()`: This initializing method directly takes in a `PySpark.sql` dataframe
- `.from_csv()`: This class method creates an instance of our custom class while reading in the data from a csv file
    - In addition to the file path, we need to provide the Spark session information
- `.from_pddf()`: This class method creates an instance of our custom class from a `pandas` dataframe
    - In addition to the dataframe name, we need to provide the Spark session information

Of course, to utilize any of these three methods, we need to make the class accessible in our environment. Let's do that now. 

*Note: The `air.csv` and `SparkDataCheck.py` files will need to be available in the reader's environment to run all code chunks in Part I.*

In [4]:
from SparkDataCheck import *

Let's utilize the `.from_csv()` class method to initialize our custom class while reading in a csv file of air pollution readings. This dataset was developed to test the efficacy of using low-cost chemical sensors to estimate true pollutant concentrations in situations where pollution monitoring systems are cost-prohibitive. Thus, the data contains hourly true concentrations and sensor readings for multiple pollutants in a single location. We will not explore the context further as our focus is on exploring the custom class. However, interested readers can access the data [here](https://archive.ics.uci.edu/dataset/360/air+quality) and learn more about the study [here](https://www.sciencedirect.com/science/article/abs/pii/S0925400507007691). 

In [6]:
# Creating a SparkDataCheck object containing the air quality data
test_data_sql = SparkDataCheck.from_csv(spark, "air.csv")

Let's print the first 25 rows of the dataset to gain a better understanding of the data structure. 

In [8]:
# Extracting the first 25 rows of the data
test_data_sql.df.show(25)

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|  1|3/10/2004|2026-03-20 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|  2|3/10/2004|2026-03-20 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|  3|3/10/2004|2026-03-20 21:00:00|   2.2|       137

26/03/20 21:53:24 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-jccrews@ncsu.edu/air.csv


Note the warning, which indicates the first column in the file does not have a name. This is simply the row index, so the warning is not concerning. 

The rows printed above indicate an idiosyncracy of the dataset that we will need to handle before we explore the methods of our custom class: missing values are coded -200.  Let's loop over the atmospheric measurements (`CO(GT)` through `AH`) to convert each `-200` to `NULL`. In the process, we will update the variable names because the "." in some of the names will be problematic for `PySpark`. 

In [16]:
# Specifying clean variable names
clean_names = ['Index', 'Date', 'Time', 'CO_true', 'CO_sensor', 'NMHC_true', 'C6H6_true', 
               'NMHC_sensor', 'NOx_true', 'NOx_sensor', 'NO2_true', 'NO2_sensor', 'O3_sensor', 
               'Temp', 'Rel_Humid', 'Abs_Humid']

# Updating names and converting the missing values (labeled -200) to NULL
for i in range(len(test_data_sql.df.columns)):
    # Updating names
    test_data_sql.df = test_data_sql.df.withColumnRenamed(test_data_sql.df.columns[i], clean_names[i])
    
    # Converting missing values for the measurements
    if i >= 3:
        test_data_sql.df = test_data_sql.df.withColumn(clean_names[i], F.when(test_data_sql.df[clean_names[i]] == -200, None).otherwise(test_data_sql.df[clean_names[i]]))        

Let's print out the first 25 rows again to ensure our updates worked. 

In [17]:
# Extracting the first 25 rows of the data
test_data_sql.df.show(25)

+-----+---------+-------------------+-------+---------+---------+---------+-----------+--------+----------+--------+----------+---------+----+---------+---------+
|Index|     Date|               Time|CO_true|CO_sensor|NMHC_true|C6H6_true|NMHC_sensor|NOx_true|NOx_sensor|NO2_true|NO2_sensor|O3_sensor|Temp|Rel_Humid|Abs_Humid|
+-----+---------+-------------------+-------+---------+---------+---------+-----------+--------+----------+--------+----------+---------+----+---------+---------+
|    0|3/10/2004|2026-03-20 18:00:00|    2.6|     1360|      150|     11.9|       1046|     166|      1056|     113|      1692|     1268|13.6|     48.9|   0.7578|
|    1|3/10/2004|2026-03-20 19:00:00|    2.0|     1292|      112|      9.4|        955|     103|      1174|      92|      1559|      972|13.3|     47.7|   0.7255|
|    2|3/10/2004|2026-03-20 20:00:00|    2.2|     1402|       88|      9.0|        939|     131|      1140|     114|      1555|     1074|11.9|     54.0|   0.7502|
|    3|3/10/2004|2026-

26/03/20 22:45:25 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-jccrews@ncsu.edu/air.csv


We've now made all the updates we need to explore `SparkDataCheck`'s methods. We'll be doing so with variables names that are easier to interpret for readers who are familiar with chemical symbols. For example, the `CO_true` column contains the true carbon monoxide concentrations. 

Let's start with the `in_range()` method, which adds a new column to the dataframe indicating whether a numeric variable in the dataset is between the `lower` and `upper` inputs, inclusive. At least one of these bounding inputs must be provided. 

For our first use of `in_range()`, let's identify which true carbon monoxide concentrations are between 1 and 3. 

In [18]:
# Adding new boolean columns indicating whether CO_true between 1 and 3
test_data_sql.in_range("CO_true", 1, 3).show(25)

+-----+---------+-------------------+-------+---------+---------+---------+-----------+--------+----------+--------+----------+---------+----+---------+---------+--------------+
|Index|     Date|               Time|CO_true|CO_sensor|NMHC_true|C6H6_true|NMHC_sensor|NOx_true|NOx_sensor|NO2_true|NO2_sensor|O3_sensor|Temp|Rel_Humid|Abs_Humid|CO_true_in_1_3|
+-----+---------+-------------------+-------+---------+---------+---------+-----------+--------+----------+--------+----------+---------+----+---------+---------+--------------+
|    0|3/10/2004|2026-03-20 18:00:00|    2.6|     1360|      150|     11.9|       1046|     166|      1056|     113|      1692|     1268|13.6|     48.9|   0.7578|          true|
|    1|3/10/2004|2026-03-20 19:00:00|    2.0|     1292|      112|      9.4|        955|     103|      1174|      92|      1559|      972|13.3|     47.7|   0.7255|          true|
|    2|3/10/2004|2026-03-20 20:00:00|    2.2|     1402|       88|      9.0|        939|     131|      1140|   

26/03/20 22:58:58 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-jccrews@ncsu.edu/air.csv


When we look at the 25 observations we are becoming rather familiar with, we now see a column of "true" and "false" values. I must say, this new column has a great name! 

One thing we should note: when the input column is `NULL`, the new column is also `NULL`. This will be convenient if we want to count true and false values without worrying about including missing values. 

Next, let's generate a new column that indicates when `CO_true` is greater than or equal to 1, with no upper bound. 